# Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.rasters import load_raster
from uavloc.plotting.flight import plot_arm_errors, plot_arm_tracks
from uavloc.simulation.arms import FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.simulation.sequence import load_flight_set, run_flights
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
CONFIGS = {   # name: (flights csv, fixes csv, segmenter, head, temperature)
    "control": ("artifacts/flights-033.1.csv", "artifacts/fixes-033.1.csv",
                "experiments/033-class-loss-in-system/artifacts/segmenter-control/model-bn-satellite.pt",
                "experiments/033-class-loss-in-system/artifacts/head-control.pt", 0.5),
    "building": ("artifacts/flights-033.2.csv", "artifacts/fixes-033.2.csv",
                 "experiments/033-class-loss-in-system/artifacts/segmenter-building/model-bn-satellite.pt",
                 "experiments/033-class-loss-in-system/artifacts/head-building.pt", 0.5),
    "structure": ("artifacts/flights-033.3.csv", "artifacts/fixes-033.3.csv",
                  "experiments/033-class-loss-in-system/artifacts/segmenter-structure/model-bn-satellite.pt",
                  "experiments/033-class-loss-in-system/artifacts/head-structure.pt", 0.5),
    "noise": ("artifacts/flights-033.4.csv", "artifacts/fixes-033.4.csv",
              "experiments/033-class-loss-in-system/artifacts/segmenter-noise/model-bn-satellite.pt",
              "experiments/033-class-loss-in-system/artifacts/head-noise.pt", 0.5),
    "027.12": ("../027-sequence-filters/artifacts/flights-027.12.csv", "../027-sequence-filters/artifacts/fixes-027.12.csv",
               "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
               "experiments/023-learned-correlation-head/artifacts/head-023.3.pt", 0.5),
}
COLOURS = {"control": "#2a78d6", "building": "#eb6834", "structure": "#1baf7a", "noise": "#9b59b6", "027.12": "#555555"}
TAIL_M = 20.0
ERROR_THRESHOLDS_M = [5.0, 10.0, 20.0]
FLIGHT_FEATURES = ["distance_m", "skipped_share", "mean_coverage", "truth_in_window_share", "truth_in_ellipse_share"]
REPLAY = [("rokety-50p2212-24p3470", 1), ("zarichanka-48p8887-26p3812", 1),        # far off in every config
          ("kryve-50p1899-23p8279", 1), ("bushcha-50p2422-25p8305", 1),            # flip between configs
          ("zastavie-49p4469-25p7044", 0), ("ruda-48p5525-26p5715", 0)]
REPLAY_TOLERANCE_M = 0.5
SEED = 42
DEVICE = "cpu"                    # CPU-only process (bug-121)
ROUTE_SET = "sim-flights"
SPLIT = "test"
INITIAL_ERROR_M = 0.0
SIGMA0_M = 1.0
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp", "seg_channels": "all", "drone_seg_gsd_m": 0.385,
    "match_gsd_m": 0.385, "matching": "weighted-ncc", "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5],
    "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0, "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
FILTER = "particles"
Q_POS_M_PER_SQRT_S = 2.0
N_PARTICLES = 2000
K_MAX_HYPOTHESES = 5
PEAKS_PER_HYPOTHESIS = 3
PEAK_NMS_M = 10.0
MERGE_DISTANCE_M = 5.0
CHILD_FLOOR_SIGMA_M = 1.0
WINDOW_SIGMAS = 3.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
HEAD_ENCODER_CHANNELS = (32, 32, 16)
HEAD_REFINE_CHANNELS = 16
FIGURES = Path("figures/tails")

# Data

In [ ]:
set_seed(SEED)
flights = {name: pd.read_csv(cfg[0]) for name, cfg in CONFIGS.items()}
fixes = {name: pd.read_csv(cfg[1]) for name, cfg in CONFIGS.items()}
{name: len(table) for name, table in flights.items()}

# Run

In [ ]:
def distribution_row(errors: pd.Series) -> dict:
    tail = errors > TAIL_M
    return {"mean_m": errors.mean(), "median_m": errors.median(), "p75_m": errors.quantile(0.75),
            "p90_m": errors.quantile(0.9), "max_m": errors.max(),
            **{f"share_under_{t:g}m": (errors < t).mean() for t in ERROR_THRESHOLDS_M},
            "tail_flights": int(tail.sum()), "tail_share_of_total_error": errors[tail].sum() / errors.sum(),
            "mean_without_tail_m": errors[~tail].mean()}

In [ ]:
distribution = pd.DataFrame({name: distribution_row(t["end_error_m"]) for name, t in flights.items()}).T
distribution.round(3)

In [ ]:
end_errors = pd.concat({name: t.set_index(["village", "route_index"])["end_error_m"] for name, t in flights.items()},
                       axis=1)
end_errors["configs_in_tail"] = (end_errors[list(CONFIGS)] > TAIL_M).sum(axis=1)
tail_table = end_errors[end_errors["configs_in_tail"] > 0].sort_values("configs_in_tail", ascending=False)
tail_table.round(1)

In [ ]:
tail_table["configs_in_tail"].value_counts().sort_index()

In [ ]:
pooled = pd.concat([t.assign(config=name) for name, t in flights.items()], ignore_index=True)
pooled["tail"] = pooled["end_error_m"] > TAIL_M
pooled.groupby("tail")[FLIGHT_FEATURES].mean().T.round(3)

In [ ]:
blind_by_tail = pd.concat([f.merge(flights[name][["village", "route_index", "end_error_m"]], on=["village", "route_index"])
                           .assign(config=name) for name, f in fixes.items()], ignore_index=True)
blind_by_tail["tail"] = blind_by_tail["end_error_m"] > TAIL_M
blind_by_tail.groupby("tail")["decision"].value_counts(normalize=True).unstack().round(3)

In [ ]:
def filter_params(temperature: float) -> FilterParams:
    return FilterParams(temperature, Q_POS_M_PER_SQRT_S, N_PARTICLES, K_MAX_HYPOTHESES, PEAKS_PER_HYPOTHESIS,
                        PEAK_NMS_M, MERGE_DISTANCE_M, CHILD_FLOOR_SIGMA_M, WINDOW_SIGMAS, WINDOW_CAP_HALF_M,
                        WINDOW_MARGIN_M)

In [ ]:
def replay(name: str, specs: list) -> list:
    _, _, segmenter, head, temperature = CONFIGS[name]
    correlator = {**CORRELATOR, "seg_checkpoint_drone": segmenter, "seg_checkpoint_sat": segmenter}
    head_spec = HeadSpec(REPO_ROOT / head, HEAD_ENCODER_CHANNELS, HEAD_REFINE_CHANNELS)
    factories = {FILTER: estimator_factory(FILTER, filter_params(temperature))}
    outcomes = run_flights(set_config, specs, INITIAL_ERROR_M, SIGMA0_M,
                           lambda raster: build_correlator(raster, correlator, DEVICE, head_spec), factories)
    return outcomes[FILTER]

In [ ]:
set_config, all_specs = load_flight_set(ROUTE_SET, split=SPLIT)
specs = [s for s in all_specs if (s.village, s.route_index) in REPLAY]
len(specs)

In [ ]:
replays = {name: {(o.spec.village, o.spec.route_index): o for o in replay(name, specs)} for name in CONFIGS}

In [ ]:
check = pd.DataFrame([{"config": name, "village": key[0], "route_index": key[1],
                       "replayed_m": o.track["est_error_m"].iloc[-1], "logged_m": end_errors.loc[key, name]}
                      for name, by_key in replays.items() for key, o in by_key.items()])
check["matches"] = (check["replayed_m"] - check["logged_m"]).abs() <= REPLAY_TOLERANCE_M
check.round(2)

# Plots

In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)
bins = np.logspace(np.log10(0.2), np.log10(300), 30)
fig, axes = plt.subplots(len(CONFIGS), 1, figsize=(8, 1.8 * len(CONFIGS)), sharex=True)
for ax, (name, table) in zip(axes, flights.items()):
    ax.hist(table["end_error_m"], bins=bins, color=COLOURS[name])
    ax.axvline(TAIL_M, color="black", lw=0.8, ls="--")
    ax.set_ylabel(name, fontsize=9)
axes[-1].set(xscale="log", xlabel="end-point error, m")
fig.suptitle(f"End-point error per flight, 94 test flights (dashed: {TAIL_M:g} m)")
fig.tight_layout()
fig.savefig(FIGURES / "end-error-histograms.png", dpi=150)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, feature in zip(axes, ["skipped_share", "truth_in_window_share", "mean_coverage"]):
    for name, table in flights.items():
        ax.scatter(table[feature], table["end_error_m"], s=10, alpha=0.6, color=COLOURS[name], label=name)
    ax.axhline(TAIL_M, color="black", lw=0.8, ls="--")
    ax.set(yscale="log", xlabel=feature)
axes[0].set_ylabel("end-point error, m")
axes[-1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "end-error-vs-flight-features.png", dpi=150)

In [ ]:
for key in REPLAY:
    tracks = {name: replays[name][key].track for name in CONFIGS}
    flight_fixes = {name: replays[name][key].fixes for name in CONFIGS}
    basemap = load_raster({"kind": "multidate", "place": key[0], "date": set_config.map_date})
    fig, (ax_map, ax_err) = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={"width_ratios": [1.1, 1]})
    plot_arm_tracks(tracks, COLOURS, basemap, ax_map)
    plot_arm_errors(tracks, flight_fixes, COLOURS, ax_err)
    ax_map.set_title(f"{key[0]} route {key[1]}")
    ax_err.set_title("error over time (x = blind fix, skipped)")
    fig.tight_layout()
    fig.savefig(FIGURES / f"{key[0]}-{key[1]}.png", dpi=120)